In [1]:
# ============================================================
# sRFTM CPU Benchmark — local analysis
# ============================================================

import time
import sys
import torch
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.model import SRFTM
from src.math_tokenizer import MathTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CPU: {torch.get_num_threads()} threads")
print(f"MPS: {torch.backends.mps.is_available()}")
print(f"CUDA: {torch.cuda.is_available()}")

# Paths
MODEL_PATH = Path.cwd().parent / 'models' / 'srftm_v4_best.pt'
print(f"\nModel: {MODEL_PATH}")
print(f"Exists: {MODEL_PATH.exists()}")

PyTorch: 2.14.0
CPU: 4 threads
MPS: True
CUDA: False

Model: /Users/mac/Desktop/sRFTM/models/srftm_v4_best.pt
Exists: True


In [2]:
# ============================================================
# Load model + inspect
# ============================================================

import time
import torch
import os

# Device
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}\n")

# Tokenizer
tokenizer = MathTokenizer()
print(f"Vocab: {len(tokenizer)}")

# Model
t0 = time.time()

model = SRFTM(
    src_vocab_size=len(tokenizer),
    tgt_vocab_size=len(tokenizer),
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_encoder_layers=2,
    n_decoder_layers=2,
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

load_time = time.time() - t0
print(f"Load time: {load_time:.3f}s")

# Size
total_params = sum(p.numel() for p in model.parameters())
file_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"Parameters: {total_params:,}")
print(f"File size: {file_size_mb:.2f} MB")

Device: mps

Vocab: 193
Load time: 0.991s
Parameters: 1,000,001
File size: 8.73 MB


In [3]:
# ============================================================
# Inference timing
# ============================================================

import time
import torch

# Test examples
SAMPLES = [
    "x^2",
    "1/n",
    "a/b",
    "x^2 = 9",
    "sqrt(a^2 + b^2)",
    "deriv(x^3)",
    "sum n=1 inf 1/n^2",
    "integral 0 1 x dx",
    "lim x->0 sin(x)/x",
    "f(x) = x^2 + 2x + 1",
]

def benchmark(func, samples, n_warmup=2, n_runs=5):
    """Усредняем время по n_runs."""
    # Warmup
    for _ in range(n_warmup):
        for s in samples:
            src = torch.tensor([tokenizer.encode(s)], device=device)
            func(src)
    
    # Sync MPS
    if device == 'mps':
        torch.mps.synchronize()
    
    # Timed
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        for s in samples:
            src = torch.tensor([tokenizer.encode(s)], device=device)
            func(src)
        if device == 'mps':
            torch.mps.synchronize()
        times.append(time.time() - t0)
    
    return times

# Greedy
def greedy_fn(src):
    return model.greedy_decode(
        src, tokenizer.sos_id, tokenizer.eos_id,
        max_len=100, repetition_penalty=1.3,
    )

# Beam
def beam_fn(src):
    return model.beam_search(
        src, tokenizer.sos_id, tokenizer.eos_id,
        max_len=100, beam_width=5,
        length_penalty=0.6, repetition_penalty=1.3,
    )

print("=== GREEDY ===")
greedy_times = benchmark(greedy_fn, SAMPLES)
avg = sum(greedy_times) / len(greedy_times)
print(f"  Total for 10 samples: {avg*1000:.0f}ms")
print(f"  Per sample: {avg/len(SAMPLES)*1000:.0f}ms")

print("\n=== BEAM (width=5) ===")
beam_times = benchmark(beam_fn, SAMPLES)
avg = sum(beam_times) / len(beam_times)
print(f"  Total for 10 samples: {avg*1000:.0f}ms")
print(f"  Per sample: {avg/len(SAMPLES)*1000:.0f}ms")

=== GREEDY ===
  Total for 10 samples: 2980ms
  Per sample: 298ms

=== BEAM (width=5) ===
  Total for 10 samples: 54044ms
  Per sample: 5404ms


In [4]:
# ============================================================
# MPS vs CPU comparison
# ============================================================

import time
import torch

def run_benchmark(dev, model, tokenizer, method='greedy'):
    model = model.to(dev)
    
    times = []
    for _ in range(3):
        if dev == 'mps':
            torch.mps.synchronize()
        
        t0 = time.time()
        for s in SAMPLES:
            src = torch.tensor([tokenizer.encode(s)], device=dev)
            if method == 'greedy':
                model.greedy_decode(src, tokenizer.sos_id, tokenizer.eos_id, max_len=100, repetition_penalty=1.3)
            else:
                model.beam_search(src, tokenizer.sos_id, tokenizer.eos_id, max_len=100, beam_width=5)
        
        if dev == 'mps':
            torch.mps.synchronize()
        times.append(time.time() - t0)
    
    return sum(times) / len(times)

print("=== GREEDY ===")
mps_time = run_benchmark('mps', model, tokenizer, 'greedy')
cpu_time = run_benchmark('cpu', model, tokenizer, 'greedy')
print(f"  MPS: {mps_time*1000:.0f}ms ({mps_time/len(SAMPLES)*1000:.0f}ms per sample)")
print(f"  CPU: {cpu_time*1000:.0f}ms ({cpu_time/len(SAMPLES)*1000:.0f}ms per sample)")

print("\n=== BEAM ===")
mps_time = run_benchmark('mps', model, tokenizer, 'beam')
cpu_time = run_benchmark('cpu', model, tokenizer, 'beam')
print(f"  MPS: {mps_time*1000:.0f}ms ({mps_time/len(SAMPLES)*1000:.0f}ms per sample)")
print(f"  CPU: {cpu_time*1000:.0f}ms ({cpu_time/len(SAMPLES)*1000:.0f}ms per sample)")

=== GREEDY ===
  MPS: 3169ms (317ms per sample)
  CPU: 179ms (18ms per sample)

=== BEAM ===
  MPS: 54994ms (5499ms per sample)
  CPU: 4707ms (471ms per sample)


In [5]:
# ============================================================
# Final benchmark summary
# ============================================================

import time
import torch

# Force CPU
device = 'cpu'
model = model.to(device)

# Warmup
for s in SAMPLES:
    src = torch.tensor([tokenizer.encode(s)], device=device)
    model.greedy_decode(src, tokenizer.sos_id, tokenizer.eos_id, max_len=100, repetition_penalty=1.3)

# Individual timing
print("=== GREEDY — per sample ===")
times = []
for s in SAMPLES:
    src = torch.tensor([tokenizer.encode(s)], device=device)
    t0 = time.time()
    model.greedy_decode(src, tokenizer.sos_id, tokenizer.eos_id, max_len=100, repetition_penalty=1.3)
    elapsed = time.time() - t0
    times.append(elapsed)
    print(f"  {s!r:40} {elapsed*1000:6.1f}ms")

print(f"\n  Avg: {sum(times)/len(times)*1000:.1f}ms")
print(f"  Min: {min(times)*1000:.1f}ms")
print(f"  Max: {max(times)*1000:.1f}ms")

# Memory
import psutil
import os
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / (1024 * 1024)
print(f"\n  Process memory: {mem_mb:.1f} MB")

=== GREEDY — per sample ===
  'x^2'                                      18.0ms
  '1/n'                                       7.9ms
  'a/b'                                       7.4ms
  'x^2 = 9'                                  10.1ms
  'sqrt(a^2 + b^2)'                          18.4ms
  'deriv(x^3)'                               15.9ms
  'sum n=1 inf 1/n^2'                        25.5ms
  'integral 0 1 x dx'                        18.8ms
  'lim x->0 sin(x)/x'                        26.8ms
  'f(x) = x^2 + 2x + 1'                      23.8ms

  Avg: 17.3ms
  Min: 7.4ms
  Max: 26.8ms

  Process memory: 57.2 MB


In [6]:
# ============================================================
# Custom examples
# ============================================================

def predict(text, method='greedy'):
    src = torch.tensor([tokenizer.encode(text)], device=device)
    
    if method == 'greedy':
        out = model.greedy_decode(
            src, tokenizer.sos_id, tokenizer.eos_id,
            max_len=100, repetition_penalty=1.3,
        )
        return tokenizer.decode(out[0].tolist(), skip_special=True)
    else:
        out = model.beam_search(
            src, tokenizer.sos_id, tokenizer.eos_id,
            max_len=100, beam_width=5,
            length_penalty=0.6, repetition_penalty=1.3,
        )
        return tokenizer.decode(out.tolist(), skip_special=True)


# Custom examples
examples = [
    # Простые
    "x^3",
    "y^2",
    "2/n",
    "5/7",
    "sqrt(x)",
    
    # Средние
    "a^2 + b^2",
    "sin(x)",
    "cos(2x)",
    "ln(x)",
    "log(x + 1)",
    
    # Сложные
    "deriv(sin(x))",
    "integral 0 pi sin(x) dx",
    "sum i=1 n i^2",
    "lim x->inf 1/x",
    "prod i=1 n i",
    
    # Уравнения
    "x^2 + y^2 = r^2",
    "a^2 - b^2 = (a-b)(a+b)",
    "sin^2(x) + cos^2(x) = 1",
    
    # ML
    "MSE",
    "accuracy = (TP + TN) / (TP + TN + FP + FN)",
    "precision = TP / (TP + FP)",
    
    # Свободные
    "x squared plus y squared",
    "1 over n plus 1 over m",
    "корень из x",
    "производная от x в кубе",
    "сумма от i равно 1 до n",
]

print("=" * 70)
print("sRFTM — Custom Examples")
print("=" * 70)

for ex in examples:
    result = predict(ex, method='greedy')
    print(f"\nInput:  {ex!r}")
    print(f"Output: {result!r}")

sRFTM — Custom Examples

Input:  'x^3'
Output: 'x^{3}x^{3}'

Input:  'y^2'
Output: '^{2}y^{2}'

Input:  '2/n'
Output: '\\frac{2}{n}'

Input:  '5/7'
Output: '\\frac{5}{75}'

Input:  'sqrt(x)'
Output: '\\sqrt{x}'

Input:  'a^2 + b^2'
Output: 'a^{2} + b^{2} + c^{2}'

Input:  'sin(x)'
Output: '\\sin(x)'

Input:  'cos(2x)'
Output: '\\cos(2x)'

Input:  'ln(x)'
Output: '\\ln(x)'

Input:  'log(x + 1)'
Output: '\\log(x+1)'

Input:  'deriv(sin(x))'
Output: '\\frac{d}{dx}(\\sin(x))'

Input:  'integral 0 pi sin(x) dx'
Output: '\\int_{0}^{\\pi} \\sin(x) \\, dx'

Input:  'sum i=1 n i^2'
Output: '\\sum_{i=1}^{n} \\mu^{2}'

Input:  'lim x->inf 1/x'
Output: '\\lim_{x \\to \\infty} \\frac{1}{x}'

Input:  'prod i=1 n i'
Output: '\\nabla \\cdot = 1}'

Input:  'x^2 + y^2 = r^2'
Output: 'x^{2} + y^{2} = r^{2}'

Input:  'a^2 - b^2 = (a-b)(a+b)'
Output: 'a^{2} - b^{2} = (a-b)(a+b)'

Input:  'sin^2(x) + cos^2(x) = 1'
Output: '\\sin^{2}(x) + \\cos^{2}(x) = 1'

Input:  'MSE'
Output: '\\sum e^{i\\omega}'

Input: 

In [7]:
from src.postprocess import postprocess

tests = [
    "x^{3}x^{3}",
    "\\frac{a}{b}\\frac{a}{b}",
    "y^{2}y^{2}y^{2}",
    "x^{3}",
    "\\frac{5}{75}",
    "a^{2} + b^{2} + c^{2}",
    "\\sqrt{x}\\sqrt{x}",
    "x_{1}x_{1}",
    "\\frac{1}{n}",
    "{x^{2}",
    "x^{2}}",
]

for t in tests:
    result = postprocess(t)
    changed = "✓" if result != t else " "
    print(f"{changed} {t!r:35} → {result!r}")

✓ 'x^{3}x^{3}'                        → 'x^{3}'
✓ '\\frac{a}{b}\\frac{a}{b}'          → '\\frac{a}{b}'
✓ 'y^{2}y^{2}y^{2}'                   → 'y^{2}'
  'x^{3}'                             → 'x^{3}'
  '\\frac{5}{75}'                     → '\\frac{5}{75}'
  'a^{2} + b^{2} + c^{2}'             → 'a^{2} + b^{2} + c^{2}'
✓ '\\sqrt{x}\\sqrt{x}'                → '\\sqrt{x}'
✓ 'x_{1}x_{1}'                        → 'x_{1}'
  '\\frac{1}{n}'                      → '\\frac{1}{n}'
✓ '{x^{2}'                            → '{x^{2}}'
✓ 'x^{2}}'                            → 'x^{2}'


In [8]:
# ============================================================
# Custom examples + postprocess
# ============================================================

import sys
sys.path.insert(0, '.')

from src.postprocess import postprocess


def predict(text, method='greedy', use_postprocess=True):
    src = torch.tensor([tokenizer.encode(text)], device=device)
    
    if method == 'greedy':
        out = model.greedy_decode(
            src, tokenizer.sos_id, tokenizer.eos_id,
            max_len=100, repetition_penalty=1.3,
        )
        decoded = tokenizer.decode(out[0].tolist(), skip_special=True)
    else:
        out = model.beam_search(
            src, tokenizer.sos_id, tokenizer.eos_id,
            max_len=100, beam_width=5,
            length_penalty=0.6, repetition_penalty=1.3,
        )
        decoded = tokenizer.decode(out.tolist(), skip_special=True)
    
    raw = decoded
    if use_postprocess:
        decoded = postprocess(decoded)
    
    return raw, decoded


examples = [
    "x^3",
    "y^2",
    "2/n",
    "5/7",
    "sqrt(x)",
    "a^2 + b^2",
    "sin(x)",
    "cos(2x)",
    "ln(x)",
    "log(x + 1)",
    "deriv(sin(x))",
    "integral 0 pi sin(x) dx",
    "sum i=1 n i^2",
    "lim x->inf 1/x",
    "x^2 + y^2 = r^2",
    "a^2 - b^2 = (a-b)(a+b)",
    "sin^2(x) + cos^2(x) = 1",
    "precision = TP / (TP + FP)",
    "x squared plus y squared",
    "1 over n plus 1 over m",
]

print("=" * 90)
print("sRFTM — Examples with postprocess")
print("=" * 90)

for ex in examples:
    raw, fixed = predict(ex, method='greedy', use_postprocess=True)
    changed = "★" if raw != fixed else " "
    print(f"\n{changed} Input:    {ex!r}")
    print(f"  Raw:      {raw!r}")
    if changed == "★":
        print(f"  Fixed:    {fixed!r}")

sRFTM — Examples with postprocess

★ Input:    'x^3'
  Raw:      'x^{3}x^{3}'
  Fixed:    'x^{3}'

  Input:    'y^2'
  Raw:      '^{2}y^{2}'

  Input:    '2/n'
  Raw:      '\\frac{2}{n}'

  Input:    '5/7'
  Raw:      '\\frac{5}{75}'

  Input:    'sqrt(x)'
  Raw:      '\\sqrt{x}'

  Input:    'a^2 + b^2'
  Raw:      'a^{2} + b^{2} + c^{2}'

  Input:    'sin(x)'
  Raw:      '\\sin(x)'

  Input:    'cos(2x)'
  Raw:      '\\cos(2x)'

  Input:    'ln(x)'
  Raw:      '\\ln(x)'

  Input:    'log(x + 1)'
  Raw:      '\\log(x+1)'

  Input:    'deriv(sin(x))'
  Raw:      '\\frac{d}{dx}(\\sin(x))'

  Input:    'integral 0 pi sin(x) dx'
  Raw:      '\\int_{0}^{\\pi} \\sin(x) \\, dx'

  Input:    'sum i=1 n i^2'
  Raw:      '\\sum_{i=1}^{n} \\mu^{2}'

  Input:    'lim x->inf 1/x'
  Raw:      '\\lim_{x \\to \\infty} \\frac{1}{x}'

  Input:    'x^2 + y^2 = r^2'
  Raw:      'x^{2} + y^{2} = r^{2}'

  Input:    'a^2 - b^2 = (a-b)(a+b)'
  Raw:      'a^{2} - b^{2} = (a-b)(a+b)'

  Input:    'sin^2(x) + 

In [13]:
import sys
if 'src.templates' in sys.modules:
    del sys.modules['src.templates']

from src.templates import postprocess, build_template
print("OK")

OK


In [ ]:
import sys
if 'src.templates' in sys.modules:
    del sys.modules['src.templates']

from src.templates import build_template

examples = [
    "x^2", "x^3", "y^2", "x^5",
    "1/2", "5/7", "2/3", "100/10",
    "a/b", "x/y",
    "sqrt(x)", "sqrt(2)", "sqrt(a^2 + b^2)",
    "cbrt(x)",
    "alpha + beta", "theta + phi",
    "a^2 + b^2", "x^2 + y^2",
    "log(x)", "ln(x)", "log_2(x)",
    "sin(x)", "cos(x)", "tan(x)", "sin(2x)",
    "deriv(x^3)", "deriv(sin(x))",
    "integral 0 1 x dx",
    "integral 0 inf e^(-x^2) dx",
    "sum i=1 n i^2",
    "sum n=1 inf 1/n^2",
    "lim x->0 sin(x)/x",
    "lim x->inf 1/x",
    "E = m c^2",
    "a + b", "a - b", "a * b",
]

for ex in examples:
    t = build_template(ex)
    status = "✓" if t else "✗"
    print(f"{status} {ex!r:35} → {t!r}")

Template-based postprocess
✓ 'x^2'                     → 'x^{2}'
✓ 'x^3'                     → 'x^{3}'
✓ 'y^2'                     → 'y^{2}'
✓ 'x^5'                     → 'x^{5}'
✓ '1/2'                     → '\\frac{1}{2}'
✓ '5/7'                     → '\\frac{5}{7}'
✓ '2/3'                     → '\\frac{2}{3}'
✓ '100/10'                  → '\\frac{100}{10}'
✓ 'a/b'                     → '\\frac{a}{b}'
✓ 'x/y'                     → '\\frac{x}{y}'
✓ 'sqrt(x)'                 → '\\sqrt{x}'
✓ 'sqrt(2)'                 → '\\sqrt{2}'
✓ 'alpha + beta'            → '\\alpha + \\beta'
✓ 'theta + phi'             → '\\theta + \\phi'
✓ 'a^2 + b^2'               → 'a^{2} + b^{2}'
✓ 'x^2 + y^2'               → 'x^{2} + y^{2}'
✓ 'log(x)'                  → '\\log(x)'
✓ 'ln(x)'                   → '\\ln(x)'
✓ 'sin(x)'                  → '\\sin(x)'
✓ 'cos(x)'                  → '\\cos(x)'
✗ 'deriv(x^3)'              → None
✗ 'integral 0 1 x dx'       → None
✗ 'sum i=1 n i^2'           → None
